## First Look transient targets

In this notebook we will:

1. Load good DIA lightcurves from DP2.
2. Crossmatch a handful of First Look targets to find matches in DP2.
3. Inspect visually that the matches are transient.
4. Generate the embeddings for the matches and save them to disk.

In [1]:
import lsdb
import math
import matplotlib.pyplot as plt
import pandas as pd

### 1. Load good DIA lightcurves

Load DIA objects:

In [2]:
dia_object = lsdb.open_catalog('/astro/store/shire/hats/dash/hats/v30_0_6/dia_object_collection')

Filter out the bad detections (those with flags):

In [3]:
flag_cols = [c for c in dia_object.meta["diaSource"].columns if 'flag' in c.lower()]
query_str = "not (" + " or ".join(f"diaSource.{c}" for c in flag_cols) + ")"
dia_object = dia_object.query(query_str).query("diaSource.len() > 100")

### 2. Find First Look targets in DP2

Create a catalog for our first look targets:

In [4]:
from io import StringIO

targets = lsdb.from_dataframe(
    pd.read_csv(
        StringIO(
            """ra dec
            187.4565 8.213469
            186.016373037 8.4117596341
            149.261451 1.291222
            51.620551 -28.114117
            10.581959 -45.278411
            10.918621 -44.157346
            10.178225 -45.842358
            52.718802 -28.362480
            61.964820 -48.713443
            52.786108 -27.341361
            """
        ), 
        sep=r'\s+'
    )
)

Then find the matches for these transients in DP2:

In [5]:
matches = targets.crossmatch(dia_object, suffixes=("", "_dp2"))
matches = matches["ra", "dec", "diaObjectId_dp2", "diaSource_dp2"].compute()
matches

### 3. Plot lightcurves of DP2 matches

Let's plot the lightcurves and visually inspect that these objects are variable.

In [6]:
matches["diaSource_dp2"]

In [ ]:
from lsdb_rubin.plot_light_curve import plot_light_curve

n_cols = 2
n_rows = math.ceil(len(matches) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = axes.flatten()

for i, match in enumerate(matches.itertuples()):
    plt.sca(axes[i])  # set current axes
    title = f"{i+1}. Match for RA={match.ra:.3f}, DEC={match.dec:.3f}\ndiaObjectId={match.diaObjectId_dp2}"
    plot_light_curve(match.diaSource_dp2, title=title)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Add red titles for transients candidates
for k in [1, 3, 4, 6, 8]:
    axes[k-1].title.set_color('red')

plt.tight_layout()
plt.show()

Matches 1, 3, 4, 6, 8 are good transient candidates.

### 2. Calculate lightcurve embeddings

Using the `light_curve` package. ATCAT processes all six LSST ugrizY bands jointly and returns 384-dimensional embeddings. Inputs are flux (AB, zero-point 31.4 by default), flux error, time, and integer band index (u=0, g=1, r=2, i=3, z=4, Y=5).

In [18]:
# Testing for a single light curve
from light_curve.embed import ATCAT

model = ATCAT.from_hf(output="last", band_groups={"u": 0, "g": 1, "r": 2, "i": 3, "z": 4, "Y": 5})

lc = matches["diaSource_dp2"].iloc[5]
time = lc["midpointMjdTai"]
flux = lc["psfFlux"]
flux_err = lc["psfFluxErr"]
band = lc["band"]

embedding = model(time, flux, flux_err, band)

# (n_bands, n_subsamples, seq_windows, embed_dim)
print(embedding.shape)  # (1, 1, 1, 384)

Now let's generate the embeddings for all the desired objects:

In [19]:
from light_curve.embed import ATCAT

def compute_embeddings(time, flux, flux_err, band):
    """Calculate embeddings for a lightcurve"""
    model = ATCAT.from_hf(output="last", band_groups={"u": 0, "g": 1, "r": 2, "i": 3, "z": 4, "y": 5})
    embeddings = model(time, flux, flux_err, band)
    return {"embeddings.value": embeddings.flatten()}

target_embeddings = matches.map_rows(
    compute_embeddings,
    columns=[f"diaSource_dp2.{col}" for col in ["midpointMjdTai","psfFlux","psfFluxErr","band"]], 
    row_container="args",
    append_columns=True,
)
target_embeddings

Write the results to a parquet on disk:

In [ ]:
target_embeddings.to_parquet("outputs/target_embeddings.parquet")

In the next notebook we will calculate the embeddings for an entire DDF (Deep Drilling Field) and compare CPU vs GPU performance.